# Jour 3 - Feature Engineering
Toutes les variables sont calculées ligne par ligne (aucune statistique globale) pour éviter le Data Leakage.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
%matplotlib inline
from src.cleaning.clean_data import load_clean
from src.features.build_features import build_features, feature_groups, LEAKAGE_RISK, FEATURE_JUSTIFICATIONS
df = load_clean()
feats = build_features(df)
print(df.shape, '->', feats.shape)

(2000, 14) -> (2000, 49)


## Aperçu des nouvelles variables

In [2]:
new_cols = [c for c in feats.columns if c not in df.columns and not c.startswith('genre_')]
feats[['title'] + new_cols].head(10)

,title,release_year,release_month,release_quarter,release_decade,month_sin,month_cos,n_genres,n_keywords,runtime_category,has_budget,budget_log,is_english,has_overview,overview_word_count,overview_char_len,title_word_count
0,Star Wars,1977,5,2,1970,5.000000e-01,-8.660254e-01,3,20,long,1,16.213406,1,1,52,312,2
1,Finding Nemo,2003,5,2,2000,5.000000e-01,-8.660254e-01,3,23,moyen,1,18.358805,1,1,53,333,2
2,Forrest Gump,1994,6,2,1990,1.224647e-16,-1.000000e+00,3,32,long,1,17.822844,1,1,42,238,2
3,American Beauty,1999,9,3,1990,-1.000000e+00,-1.836970e-16,1,36,long,1,16.523561,1,1,26,175,2
4,Citizen Kane,1941,4,2,1940,8.660254e-01,-5.000000e-01,2,16,moyen,1,13.640833,1,1,51,285,2
5,The Fifth Element,1997,5,2,1990,5.000000e-01,-8.660254e-01,3,32,long,1,18.315320,0,1,28,145,3
6,Pirates of the Caribbean: The Curse of the Bla...,2003,7,3,2000,-5.000000e-01,-8.660254e-01,3,19,long,1,18.757153,1,1,46,312,10
7,Kill Bill: Vol. 1,2003,10,4,2000,-8.660254e-01,5.000000e-01,2,14,moyen,1,17.216708,1,1,24,138,4
8,9 Songs,2004,7,3,2000,-5.000000e-01,-8.660254e-01,3,34,court,1,13.815512,1,1,69,408,2
9,Apocalypse Now,1979,5,2,1970,5.000000e-01,-8.660254e-01,2,26,long,1,17.265498,1,1,57,333,2


## Distribution des variables créées

In [3]:
feats[['n_genres','n_keywords','overview_word_count','budget_log','release_year']].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
n_genres,2000.0,2.77,0.95,1.00,2.00,3.00,3.00,6.00
n_keywords,2000.0,15.71,8.85,0.00,10.00,15.00,20.00,98.00
overview_word_count,2000.0,45.62,20.06,8.00,29.00,42.00,58.00,137.00
budget_log,1717.0,17.46,1.34,11.16,16.76,17.69,18.42,20.31
release_year,2000.0,2011.25,14.88,1933.00,2004.00,2015.00,2023.00,2026.00


In [4]:
feats['runtime_category'].value_counts(dropna=False)

runtime_category
moyen        1145
long          537
court         181
tres_long     134
NaN             3
Name: count, dtype: int64

## Groupes de variables (pour le ColumnTransformer)

In [5]:
groups = feature_groups(feats)
{k: len(v) for k, v in groups.items()}

{'numeric': 10,
 'binary': 22,
 'categorical': 2,
 'descriptive_only': 3,
 'text': 1,
 'leakage_risk': 4,
 'identifiers': 2}

## Variables écartées : risque de Data Leakage

In [6]:
pd.Series(LEAKAGE_RISK, name='raison').to_frame()

,raison
vote_count,Sert à construire la cible high_engagement : i...
vote_average,Résultat direct de l'engagement du public (les...
popularity,"Score TMDB calculé à partir des vues, votes et..."
revenue,"Conséquence du succès du film, connue seulemen..."


## Valeurs manquantes des features (traitées plus tard dans le Pipeline)

In [7]:
feats[groups['numeric'] + groups['categorical']].isna().mean().mul(100).round(1).sort_values(ascending=False).head(8)

budget_log          14.2
runtime              0.2
runtime_category     0.2
release_year         0.0
n_genres             0.0
month_cos            0.0
month_sin            0.0
n_keywords           0.0
dtype: float64

## Sauvegarde
(ou `python src/features/build_features.py`)

In [8]:
from src.features.build_features import run_feature_engineering
_ = run_feature_engineering()

Films : 2000 | colonnes : 14 -> 49
  numeric           (10) : ['release_year', 'month_sin', 'month_cos', 'n_genres', 'n_keywords', 'runtime', 'budget_log', 'overview_word_count', '... (+2)']
  binary            (22) : ['has_budget', 'is_english', 'has_overview', 'genre_Action', 'genre_Adventure', 'genre_Animation', 'genre_Comedy', 'genre_Crime', '... (+14)']
  categorical       ( 2) : ['runtime_category', 'original_language']
  descriptive_only  ( 3) : ['release_month', 'release_quarter', 'release_decade']
  text              ( 1) : ['overview']
  leakage_risk      ( 4) : ['vote_count', 'vote_average', 'popularity', 'revenue']
  identifiers       ( 2) : ['movie_id', 'title']

Fichier : /home/iamjustwliya/Movie_Intelligence/data/features/movies_features.csv
Groupes : /home/iamjustwliya/Movie_Intelligence/data/features/feature_groups.json
Rapport : /home/iamjustwliya/Movie_Intelligence/reports/feature_justifications.md
